# Lab 5.2 &mdash; A Research Brief Written by Several Agents at Once

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Fan out from one planner to three researchers that run in a single superstep
- Meet <code>InvalidUpdateError</code> for real, then declare the reducer that fixes it
- Resolve a disagreement between two sources &mdash; by authority, not by vote
- Carry provenance into the report, so every line can be traced back

> **How this lab works.** You write real LangGraph code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a compiled `StateGraph`, a declared reducer, a routing key), so they are deterministic
> and never depend on the model. Cells marked **Run it for real** put your work in front of
> the sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **The system on slide 3.** A planner splits the brief, three researchers work at the
> same time, a declared reducer merges what they return, and a writer produces one report.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def _is_todo(exc: BaseException) -> bool:
    """Is this exception really an unfilled blank?

    LangGraph runs your nodes inside tasks, so the NameError from an unfilled BLANK can
    arrive wrapped. Walk the cause chain before calling anything a failure -- telling you
    your answer is wrong when you have not written one yet is the worst thing a lab does.
    """
    seen = set()
    while exc is not None and id(exc) not in seen:
        if isinstance(exc, NameError):
            return True
        seen.add(id(exc))
        exc = exc.__cause__ or exc.__context__
    return False

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except Exception as exc:
        if _is_todo(exc):
            print(f"[TODO] {name}")
            _results.append(None)
            return
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except Exception as exc:
        if not _is_todo(exc):
            raise
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because the "Run it for real" cells make many small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# A vendor due-diligence brief. Three questions, three sources each, one report.
# This case file is Module 5 lab 5.2 only -- 5.1 and 5.3 are different systems.

BRIEF = "Should we adopt Vendor X for document storage?"

# Each source carries WHERE it came from and how much weight that origin deserves.
# authority: 3 = a signed contract, 2 = a filed report, 1 = the vendor's own marketing.
SOURCES = {
    "pricing": [
        {"claim": "list price is 18 USD per seat per month", "source": "order form",  "authority": 3},
        {"claim": "20% discount above 500 seats",            "source": "order form",  "authority": 3},
        {"claim": "cheapest in its class",                   "source": "vendor site", "authority": 1},
    ],
    "security": [
        {"claim": "data is stored in the EU only",           "source": "vendor site", "authority": 1},
        {"claim": "data may be replicated to us-east-1",     "source": "signed DPA",  "authority": 3},
        {"claim": "SOC 2 Type II, audited 2026-03",          "source": "audit report", "authority": 2},
    ],
    "support": [
        {"claim": "99.9% uptime commitment",                 "source": "order form",  "authority": 3},
        {"claim": "4-hour response on P1",                   "source": "order form",  "authority": 3},
        {"claim": "24/7 human support",                      "source": "vendor site", "authority": 1},
    ],
}

# The two that cannot both be true. Data residency is the decision the whole brief turns on.
CONFLICT = ("data is stored in the EU only", "data may be replicated to us-east-1")

print(f"brief: {BRIEF}\n{sum(len(v) for v in SOURCES.values())} claims across "
      f"{len(SOURCES)} questions")

## Concept

Fanning out is easy: give three nodes the same predecessor and LangGraph runs them in one
**superstep**. Merging is the part that is a design decision.

Every one of those three nodes returns a partial state. If they all write the same key, LangGraph
needs to be told what "both" means &mdash; and if you never told it, it does **not** quietly keep
the last one. It raises `InvalidUpdateError`. Silent loss is what a hand-rolled `dict.update` does,
and a hand-rolled merge is what most people write first.

So: the framework protects the keys you declared, and only those.

## Section 1 &mdash; Fan out, and declare the merge

Two graphs, built from the same nodes. The first leaves `notes` un-annotated and is here to fail
in front of you. The second is yours to finish: pick the **reducer** that makes three parallel
writes into one list.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

QUESTIONS = ("pricing", "security", "support")


# ---- the graph that does NOT work, so you can see what it does instead of being told
class NaiveState(TypedDict):
    brief: str
    notes: list                       # no reducer -- three nodes are about to write it


def naive_researcher(topic: str):
    def node(state: NaiveState) -> dict:
        return {"notes": [c["claim"] for c in SOURCES[topic]]}
    return node


def collision() -> str:
    """Run the un-annotated fan-out and report what actually happens."""
    g = StateGraph(NaiveState)
    for q in QUESTIONS:
        g.add_node(q, naive_researcher(q))
        g.add_edge(START, q)
        g.add_edge(q, END)
    try:
        out = g.compile().invoke({"brief": BRIEF, "notes": []})
        return f"no error -- {len(out['notes'])} of 9 claims survived"
    except Exception as exc:
        return f"{type(exc).__name__}: {str(exc).splitlines()[0][:90]}"


# ---- the graph that does
def merge_findings(existing: list, incoming: list) -> list:
    """The merge rule for `findings`. A reducer is just this: (existing, incoming) -> combined.

    LangGraph calls it once per writer, so three researchers in one superstep call it three
    times. Concatenation keeps every writer's findings, in arrival order -- which is exactly
    what `operator.add` does on a list, and why you usually see `Annotated[list, add]`.
    """
    return existing + incoming


class BriefState(TypedDict):
    brief: str
    questions: list
    # Three researchers finish in the same superstep and all write this key. The merge rule
    # is declared here, once, and it holds for every future writer of the key.
    findings: Annotated[list, merge_findings]
    report: list


def plan(state: BriefState) -> dict:
    """Split the brief. Real planners ask a model; this one is fixed so the graph is exact."""
    return {"questions": list(QUESTIONS)}


RESEARCH_SECONDS = 0.15          # stands in for the reading a real researcher would do

def make_researcher(topic: str):
    def researcher(state: BriefState) -> dict:
        time.sleep(RESEARCH_SECONDS)
        return {"findings": [{"question": topic, **c} for c in SOURCES[topic]]}
    return researcher


def build_brief_graph(writer):
    g = StateGraph(BriefState)
    g.add_node("plan", plan)
    for q in QUESTIONS:
        g.add_node(q, make_researcher(q))
    g.add_node("writer", writer)

    g.add_edge(START, "plan")
    for q in QUESTIONS:
        g.add_edge("plan", q)          # fan out: one superstep, three nodes
        g.add_edge(q, "writer")        # fan in
    g.add_edge("writer", END)
    return g.compile()


def fresh_brief() -> dict:
    return {"brief": BRIEF, "questions": [], "findings": [], "report": []}

In [ ]:
# --- Self-check: Section 1   (real graphs, really running -- still no model)
def _noop_writer(state: BriefState) -> dict:
    return {}

def _collect() -> dict:
    return build_brief_graph(_noop_writer).invoke(fresh_brief())

check("the un-annotated fan-out does NOT silently drop two of the three",
      lambda: "InvalidUpdate" in collision(),
      "LangGraph refuses the ambiguous write; silent loss is what a hand-rolled merge does")
check("the graph with a declared merge rule compiles",
      lambda: build_brief_graph(_noop_writer) is not None)
check("all nine claims survive the merge",
      lambda: len(_collect()["findings"]) == 9,
      "the reducer decides this: without one there is no answer, with the wrong one there are three")
check("every question is represented",
      lambda: {f["question"] for f in _collect()["findings"]} == set(QUESTIONS))
check("provenance survived the merge too",
      lambda: all("source" in f and "authority" in f for f in _collect()["findings"]),
      "a merged claim you cannot trace is a claim you cannot defend")

def _supersteps():
    print(collision(), "\n")
    # `values` mode emits the state once per SUPERSTEP, not once per node -- which is how
    # you can see the three researchers land together rather than one after another.
    for i, snap in enumerate(build_brief_graph(_noop_writer).stream(fresh_brief(),
                                                                    stream_mode="values")):
        print(f"  after step {i}: {len(snap['findings'])} findings")
    t0 = time.time()
    build_brief_graph(_noop_writer).invoke(fresh_brief())
    print(f"\n  three researchers x {RESEARCH_SECONDS}s of work each, "
          f"whole graph: {time.time() - t0:.2f}s")
    print("  ^ nine findings arrive in ONE step, and the clock reads one researcher, not three.")
    print("    Note what did NOT change: the same nine claims were paid for either way.")
guard(_supersteps)

## Section 2 &mdash; The writer, and the disagreement

Two of the nine claims cannot both be true: the vendor's site says data stays in the EU, the
signed DPA says it may be replicated to `us-east-1`. Two sources say EU-ish things and one says
otherwise, so **a vote gets this wrong**.

Agreement is not truth &mdash; three researchers given the same marketing page agree confidently,
because they inherited the error rather than each finding it. Authority and provenance are
checkable; a vote is not.

In [ ]:
def resolve_conflict(claims: list[dict]) -> dict:
    """Two claims contradict each other. Return the one the report should carry.

    Given: `claims` are the contradicting findings, each already carrying `source`,
    `authority` (higher is stronger) and `agreeing` (how many researchers said it).

    Authority, not agreement. The signed DPA outranks the marketing page even when the
    marketing page is repeated more often -- repetition is not evidence.
    """
    return max(claims, key=lambda c: c["authority"])


def writer(state: BriefState) -> dict:
    """Turn the merged findings into report lines, one per claim, conflict resolved."""
    findings = state["findings"]
    contested = [f for f in findings if f["claim"] in CONFLICT]
    for f in contested:
        f["agreeing"] = sum(1 for g in findings if g["claim"] == f["claim"])
    winner = resolve_conflict(contested)["claim"] if contested else None

    lines = []
    for f in findings:
        if f["claim"] in CONFLICT and f["claim"] != winner:
            continue                                   # the losing side of the conflict
        # A line nobody can trace is a line nobody can defend, so carry the origin
        # through from the finding into the report.
        lines.append(f"{f['question']}: {f['claim']}  [{f['source']}]")
    return {"report": lines}

In [ ]:
# --- Self-check: Section 2
_EU, _US = CONFLICT

def _report():
    return build_brief_graph(writer).invoke(fresh_brief())["report"]

check("the signed DPA beats the vendor's own site",
      lambda: resolve_conflict([
          {"claim": _EU, "source": "vendor site", "authority": 1, "agreeing": 2},
          {"claim": _US, "source": "signed DPA",  "authority": 3, "agreeing": 1},
      ])["claim"] == _US,
      "authority is checkable; a show of hands is not")
check("and still beats it when the weaker claim has MORE voices",
      lambda: resolve_conflict([
          {"claim": _EU, "source": "vendor site", "authority": 1, "agreeing": 9},
          {"claim": _US, "source": "signed DPA",  "authority": 3, "agreeing": 1},
      ])["claim"] == _US,
      "if agreement decided it, three agents fed one bad page would carry the report")
check("the report keeps one side of the conflict, not both",
      lambda: sum(1 for line in _report() if _EU in line or _US in line) == 1)
check("and it is the side the contract says",
      lambda: any(_US in line for line in _report()))
check("every line names where it came from",
      lambda: all(any(f"[{s}]" in line for s in
                      ("order form", "vendor site", "signed DPA", "audit report"))
                  for line in _report()),
      "provenance is the whole reason the merge kept those fields")
check("the report is shorter than the findings by exactly the losing claim",
      lambda: len(_report()) == 8)

def _show():
    for line in _report():
        print("  " + line)
guard(_show)

## Run it for real &mdash; the writer becomes an agent

The graph does not change at all. `writer` is swapped for a node that hands the merged findings
to the model and asks for two paragraphs &mdash; and the same conflict is still in the input, so
watch whether the model resolves it the way your rule did, or splits the difference.

In [ ]:
WRITER_SYSTEM = ("You write a short vendor due-diligence note for a procurement committee. "
                 "Two paragraphs, no bullet points. Every factual sentence must name its "
                 "source in square brackets. If two sources contradict each other, say so "
                 "explicitly and follow the contractual one.")

def llm_writer(state: BriefState) -> dict:
    findings = state["findings"]
    listing = "\n".join(f"- ({f['question']}) {f['claim']}  [source: {f['source']}]"
                        for f in findings)
    note = ask(f"Question: {state['brief']}\n\nFindings:\n{listing}", system=WRITER_SYSTEM)
    return {"report": [note]}


def _write_for_real():
    out = build_brief_graph(llm_writer).invoke(fresh_brief())
    print(textwrap.fill(out["report"][0], 96))
    print("\n--- did it notice the contradiction? ---")
    body = out["report"][0].lower()
    print("  mentions replication to us-east-1:", "us-east-1" in body)
    print("  mentions the DPA               :", "dpa" in body)

if llm_ready():
    guard(_write_for_real)

In [ ]:
score()

## Your turn

1. Add a fourth researcher &mdash; `legal` &mdash; with claims that contradict `pricing`. You should
   not have to touch the reducer, the writer or the state to do it. If you did, the merge rule was
   in the wrong place.
2. Replace `max(..., key=authority)` with a real tie-break: what happens when two claims have the
   same authority? Decide, and write the check that would have caught the old behaviour.
3. Time the fan-out against the same three researchers in a row. Tokens will not move; wall clock
   will. Say out loud which budget you just spent, because they are not the same budget.